In [1]:
from collections import Counter

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

In [2]:
from imblearn.datasets import fetch_datasets

from imblearn.under_sampling import EditedNearestNeighbours, TomekLinks

In [3]:
import imblearn.over_sampling
import imblearn.combine

In [4]:
[print(item+',') for item in dir(imblearn.over_sampling) if 'smote' in item.lower()]

BorderlineSMOTE,
KMeansSMOTE,
SMOTE,
SMOTEN,
SMOTENC,
SVMSMOTE,
_smote,


[None, None, None, None, None, None, None]

In [5]:
from imblearn.over_sampling import (
    BorderlineSMOTE,
    SMOTE,
    SVMSMOTE,

)

In [6]:
[print(item+',') for item in dir(imblearn.combine) if 'smote' in item.lower()]

SMOTEENN,
SMOTETomek,
_smote_enn,
_smote_tomek,


[None, None, None, None]

In [7]:
from imblearn.combine import (
    SMOTEENN,
SMOTETomek,
)

In [8]:
oversampler_dict = {

    'smote': SMOTE(
        sampling_strategy='auto',
        random_state=0,
        k_neighbors=5,
        ),

    'border1': BorderlineSMOTE(
        sampling_strategy='auto',
        random_state=0,
        k_neighbors=5,
        m_neighbors=10,
        kind='borderline-1',
        ),

    'svm': SVMSMOTE(
        sampling_strategy='auto',
        random_state=0,
        k_neighbors=5,
        m_neighbors=10,
        svm_estimator=SVC(kernel='linear')),

    'smenn': SMOTEENN(
        sampling_strategy='auto',
        random_state=0,
        smote=SMOTE(sampling_strategy='auto', random_state=0, k_neighbors=5),
        enn=EditedNearestNeighbours(
            sampling_strategy='auto', n_neighbors=3, kind_sel='all'),
        ),

    'smtomek': SMOTETomek(
        sampling_strategy='auto',
        random_state=0,
        smote=SMOTE(sampling_strategy='auto', random_state=0, k_neighbors=5),
        tomek=TomekLinks(sampling_strategy='all'),
        ),

}

In [9]:
datasets_ls = [
    'car_eval_34',
    'ecoli',
    'thyroid_sick',
    'arrhythmia',
    'ozone_level'
]

In [10]:
for dataset in datasets_ls:
    data = fetch_datasets()[dataset]
    print(dataset)
    print(Counter(data.target))
    print()

car_eval_34
Counter({np.int64(-1): 1594, np.int64(1): 134})

ecoli
Counter({np.int64(-1): 301, np.int64(1): 35})

thyroid_sick
Counter({np.int64(-1): 3541, np.int64(1): 231})

arrhythmia
Counter({np.int64(-1): 427, np.int64(1): 25})

ozone_level
Counter({np.int64(-1): 2463, np.int64(1): 73})



In [11]:
def run_randomForests(X_train, X_test, y_train, y_test):

    rf = RandomForestClassifier(
        n_estimators=100, random_state=39, max_depth=2, n_jobs=-1
    )
    rf.fit(X_train, y_train)

    print('Train set')
    pred = rf.predict_proba(X_train)[:,1]
    print(
        'Random Forests roc-auc: {}'.format(roc_auc_score(y_train, pred))
        )
    
    print('Test set')
    pred = rf.predict_proba(X_test)[:,1]
    print(
        'Random Forests roc-auc: {}'.format(roc_auc_score(y_test, pred))
    )

    return roc_auc_score(y_test, pred)

In [15]:
results_dict = {}
shapes_dict = {}

for dataset in datasets_ls:

    results_dict[dataset] = {}
    shapes_dict[dataset] = {}

    print(dataset)

    #Load dataset
    data = fetch_datasets()[dataset]

    #separating train and test set
    X_train, X_test, y_train, y_test = train_test_split(
        data.data,
        data.target,
        test_size=0.3,
        random_state=0
    )

    #needs to scale since sampler is knn-based
    scaler = MinMaxScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)

    roc = run_randomForests(X_train, X_test, y_train, y_test)

    #Store results
    results_dict[dataset]['full_data'] = roc
    shapes_dict[dataset]['full_data'] = len(X_train)

    print()

    for oversampler in oversampler_dict.keys():
        print(oversampler)

        #Resampling
        X_res, y_res = oversampler_dict[oversampler].fit_resample(X_train, y_train)

        #Evaluate performance
        roc = run_randomForests(X_res, X_test, y_res, y_test)

        #store results
        results_dict[dataset][oversampler] = roc
        shapes_dict[dataset][oversampler] = len(X_res)
        print()

    print()    






car_eval_34
Train set
Random Forests roc-auc: 0.9581261802905924
Test set
Random Forests roc-auc: 0.9440504133074803

smote
Train set
Random Forests roc-auc: 0.9892067644300562
Test set
Random Forests roc-auc: 0.982268598836616

border1
Train set
Random Forests roc-auc: 0.9892200125897663
Test set
Random Forests roc-auc: 0.9863506480253086

svm
Train set
Random Forests roc-auc: 0.9898226031268869
Test set
Random Forests roc-auc: 0.9834677007857945

smenn
Train set
Random Forests roc-auc: 0.9891561805475264
Test set
Random Forests roc-auc: 0.9806868047759976

smtomek
Train set
Random Forests roc-auc: 0.9892067644300562
Test set
Random Forests roc-auc: 0.982268598836616


ecoli
Train set
Random Forests roc-auc: 0.9716599190283401
Test set
Random Forests roc-auc: 0.9408212560386474

smote
Train set
Random Forests roc-auc: 0.9773356837068748
Test set
Random Forests roc-auc: 0.9601449275362319

border1
Train set
Random Forests roc-auc: 0.9769121586044276
Test set
Random Forests roc-auc: 0.9